# Logistic Regression Baseline - Predict Customer Churn - Playground Series (Season 6 Episode 3)
------

This notebook presents a logistic regression pipeline to get a baseline scenario and a useful approach for creating diverse models.

## Loading data

In [13]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
pd.set_option("display.max_columns", None)

if os.getenv("KAGGLE_KERNEL_RUN_TYPE"):
    ENV = "kaggle"
    DATA_DIR = Path()
    ORIG_DIR = Path()
else:
    ENV = "local"
    DATA_DIR = Path("./data")
    ORIG_DIR = DATA_DIR
print(f"Environment: {ENV}")
train = pd.read_csv(DATA_DIR/"train.csv").drop("id", axis=1)
test = pd.read_csv(DATA_DIR/"test.csv").drop("id", axis=1)
orig_fn = "original.csv" if ENV == "local" else "WA_Fn-UseC_-Telco-Customer-Churn.csv"
original = pd.read_csv(DATA_DIR/orig_fn).drop("customerID", axis=1)
sub = pd.read_csv(DATA_DIR/"sample_submission.csv")
print("Data Loaded")

Environment: local
Data Loaded


In [11]:
import psutil
import torch
print(f"Logical Cores: {psutil.cpu_count(True)}")
print(f"Physical Cores: {psutil.cpu_count(False)}")
vm = psutil.virtual_memory().total /1e9
print(f"Total Ram: {vm:.2f}")
device = "gpu" if torch.cuda.is_available() else "cpu"
print(f"Using {device}")

Logical Cores: 16
Physical Cores: 8
Total Ram: 10.18
Using cpu


In [33]:
TARGET = "Churn"
FEATS = test.columns.tolist()
CATS = test.select_dtypes(include="object").columns.tolist()
NUMS = test.select_dtypes(include="number").columns.tolist()
DSEED = 10431053

In [14]:
for df in (train, original):
    df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})
for df in (train, test, original):
    df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})
original["TotalCharges"] = pd.to_numeric(original["TotalCharges"], errors="coerce")
original.loc[original["tenure"]==0, "TotalCharges"] = 0   

In [15]:
display(f"Train shape: {train.shape}")
display(train.head(3))
display(f"Test shape: {test.shape}")
display(test.head(3))
display(f"Original shape: {original.shape}")
display(original.head(3))

'Train shape: (594194, 20)'

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Male,No,Yes,Yes,29,Yes,No,DSL,Yes,No,Yes,Yes,No,No,One year,Yes,Mailed check,60.1,1653.85,0
1,Male,No,Yes,Yes,58,Yes,No,DSL,Yes,Yes,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.5,3778.20,0
2,Male,No,Yes,No,58,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.4,5841.35,0


'Test shape: (254655, 19)'

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Female,No,Yes,No,72,Yes,Yes,Fiber optic,Yes,Yes,Yes,Yes,Yes,Yes,Two year,Yes,Electronic check,115.55,8061.50
1,Female,No,Yes,No,71,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Bank transfer (automatic),19.80,1336.50
2,Male,No,No,No,12,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Bank transfer (automatic),55.55,633.55


'Original shape: (7043, 20)'

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,No,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,No,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1


In [48]:
from sklearn.preprocessing import (
    StandardScaler, SplineTransformer, OneHotEncoder, OrdinalEncoder, PolynomialFeatures, TargetEncoder, FunctionTransformer
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression, SGDClassifier, RidgeClassifier
import time

In [35]:
def make_basic1_prep(num_feats, cat_feats):
    return ColumnTransformer(
        [
            ("num_feats", StandardScaler(), num_feats),
            ("cat_feats", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="infrequent_if_exist"), cat_feats)
        ]
    )
make_basic1_prep(NUMS, CATS)

,transformers,"[('num_feats', ...), ('cat_feats', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,copy,True
,with_mean,True
,with_std,True


In [53]:
SAVE_DIR = Path("./save")
SAVE_DIR.mkdir(exist_ok=True)
FOLDS = 5
KF = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=DSEED)

def train_linear(X, y, test, prep_fn, model, exp_name="Logistic_reg", num_feats=NUMS, cat_feats=CATS):
    print("="*78)
    print(f" {exp_name}")
    print("="*78)

    oof = np.zeros(train.shape[0])
    test_preds = np.zeros(test.shape[0])
    fold_auc = []
    feats = num_feats + cat_feats
    t0 = time.time()
    for f, (tr_idx, va_idx) in enumerate(KF.split(X,y)):
        X_tr = X.loc[tr_idx, feats].copy()
        X_va = X.loc[va_idx, feats].copy() 
        X_test = test[feats].copy()
        y_tr = y[tr_idx]
        y_va = y[va_idx]
        prep = prep_fn(num_feats, cat_feats)
        X_tr_ts = prep.fit_transform(X_tr, y_tr)
        X_va_ts = prep.transform(X_va)
        X_te_ts = prep.transform(X_test)
        if f==0:
            print(f"    Fold train shape before prep: {X_tr.shape}")
            print(f"    Fold train shape after prep:  {X_tr_ts.shape}")
        m = clone(model)
        m.fit(X_tr_ts, y_tr)
        if hasattr(m, "predict_proba"): 
            oof[va_idx] = m.predict_proba(X_va_ts)[:,1]
            test_preds = m.predict_proba(X_te_ts)[:,1]
        elif hasattr(m, "decision_function"):
            oof[va_idx] = m.decision_function(X_va_ts)
            test_preds = m.decision_function(X_te_ts)
        auc = roc_auc_score(y_va, oof[va_idx])
        fold_auc.append(auc)
        print(f"    Fold {f+1}: AUC={auc:.6f}")
    test_preds /= FOLDS
    cv_auc = roc_auc_score(y, oof)
    elapsed = time.time() - t0       
    print(f"  CV AUC: {cv_auc:.6f} | mean: {np.mean(fold_auc):.6f} ± {np.std(fold_auc):.6f} | {elapsed:.1f}s\n")
    np.save(SAVE_DIR/f"{exp_name}_oof.npy", oof)
    np.save(SAVE_DIR/f"{exp_name}_test.npy", test_preds)
    meta ={
        "experiment_name": exp_name,
        "cv_auc": cv_auc,
        "fold_scores": fold_auc,
        "elapsed": elapsed
    }
    return oof, test_preds, meta

In [55]:
LOG = []

In [57]:
X = train.drop(TARGET, axis=1)
y = train[TARGET]
lg_reg = LogisticRegression(max_iter=50000)
oof_1, test_preds1, meta1 = train_linear(X, y, test, make_basic1_prep, lg_reg)
LOG.append(meta1)

 Logistic_reg
    Fold train shape before prep: (475355, 19)
    Fold train shape after prep:  (475355, 30)
    Fold 1: AUC=0.908524
    Fold 2: AUC=0.907801
    Fold 3: AUC=0.908389
    Fold 4: AUC=0.907674
    Fold 5: AUC=0.907351
  CV AUC: 0.907947 | mean: 0.907948 ± 0.000443 | 21.8s



In [58]:
LOG

[{'experiment_name': 'Logistic_reg',
  'cv_auc': 0.9079466789950952,
  'fold_scores': [0.9085243016691995,
   0.9078005413023973,
   0.9083889426725767,
   0.9076736550201593,
   0.9073510038929521],
  'elapsed': 21.77716374397278}]